# Insurance Linear Regression

This notebook builds a multiple linear regression model that predicts medical insurance `charges` from `age`, `bmi`, `children`, and `smoker`. Each section contains comments explaining what the code does and why the step is required.

## 1. Import the required libraries

NumPy and pandas support numerical and tabular analysis. Matplotlib and Seaborn are used for visualization. Scikit-learn provides the encoders, data-splitting function, regression model, and evaluation metrics.

In [ ]:
# Import libraries for numerical calculations and data manipulation.
import numpy as np
import pandas as pd

# Import visualization libraries for exploring results.
import matplotlib.pyplot as plt
import seaborn as sns

# Import the updated categorical-encoding tools.
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Import tools for splitting data, training linear regression, and evaluation.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Apply a consistent chart style.
sns.set_theme(style="whitegrid")

## 2. Load `insuranceData.csv`

The CSV file is loaded into a pandas DataFrame. The first records and dataset structure are displayed to confirm that the source loaded correctly.

In [ ]:
# Load the medical insurance dataset stored beside this notebook.
dataset = pd.read_csv("insuranceData.csv")

# Display the first five rows and basic dataset information.
print(dataset.head())
print("\nDataset shape:", dataset.shape)
print("\nColumn data types:")
print(dataset.dtypes)
print("\nMissing values by column:")
print(dataset.isna().sum())

## 3. Separate the independent and dependent variables

The independent variables in `X` are `age`, `bmi`, `children`, and `smoker`. The dependent variable in `y` is `charges`, which is the continuous value the regression model will predict.

In [ ]:
# Select every column except charges as the model inputs.
X = dataset.iloc[:, :-1].values

# Select charges as the numeric prediction target.
y = dataset.iloc[:, -1].values

print("Independent-variable shape:", X.shape)
print("Target-variable shape:", y.shape)

## 4. Apply `LabelEncoder()` to the categorical data

`smoker` is the only categorical field in the insurance dataset and is located at column index 3 in `X`. Label encoding first converts its text values (`no` and `yes`) into numeric labels.

In [ ]:
# Create a LabelEncoder and convert smoker text values into numeric labels.
labelencoder = LabelEncoder()
X[:, 3] = labelencoder.fit_transform(X[:, 3])

# Display the learned mapping for documentation.
smoker_mapping = dict(zip(labelencoder.classes_, labelencoder.transform(labelencoder.classes_)))
print("Smoker label mapping:", smoker_mapping)

## 5. Apply the updated `OneHotEncoder()` method

One-hot encoding creates a separate binary column for each smoker category. `ColumnTransformer` is the supported replacement for the deprecated categorical encoder used in older versions of the lesson. `remainder="passthrough"` keeps the numeric variables unchanged.

In [ ]:
# One-hot encode column index 3 and pass the other predictors through unchanged.
onehotencoder = ColumnTransformer(
    [("Smoker", OneHotEncoder(), [3])],
    remainder="passthrough",
)
X = onehotencoder.fit_transform(X)

# Convert the transformed result to a numeric NumPy array for modeling.
X = np.asarray(X, dtype=float)

print("Encoded feature shape:", X.shape)
print("Encoded feature names:", onehotencoder.get_feature_names_out().tolist())

## 6. Split the data using a 15% test set

The model trains on 85% of the records and is evaluated on the remaining 15%. A fixed random state makes the split reproducible.

In [ ]:
# Split the encoded data, reserving exactly 15% for testing.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Testing percentage:", f"{len(X_test) / len(X):.2%}")

## 7. Train the multiple linear regression model

The model learns the relationship between the encoded predictors and medical insurance charges using the training records only.

In [ ]:
# Create and train the linear regression model.
regressor = LinearRegression()
regressor.fit(X_train, y_train)

# Generate charge predictions for the unseen test records.
y_pred = regressor.predict(X_test)

## 8. Compare actual and predicted charges

A small comparison table makes it easy to inspect how closely the model's predictions match actual insurance charges.

In [ ]:
# Create a readable table containing actual values, predictions, and errors.
prediction_results = pd.DataFrame({
    "Actual Charges": y_test,
    "Predicted Charges": y_pred,
})
prediction_results["Prediction Error"] = (
    prediction_results["Actual Charges"]
    - prediction_results["Predicted Charges"]
)

print(prediction_results.head(10).round(2))

## 9. Evaluate model performance

Mean Absolute Error (MAE) measures the typical absolute prediction error. Root Mean Squared Error (RMSE) gives additional weight to large errors. R² measures the proportion of variation in charges explained by the model.

In [ ]:
# Calculate standard regression performance metrics.
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r_squared = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: ${mae:,.2f}")
print(f"Root Mean Squared Error: ${rmse:,.2f}")
print(f"R-squared: {r_squared:.4f}")

## 10. Visualize actual versus predicted charges

Points close to the diagonal reference line represent accurate predictions. Larger vertical distances from the line represent larger prediction errors.

In [ ]:
# Plot actual charges against the model's predicted charges.
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.70, color="royalblue")

# Add a perfect-prediction reference line for comparison.
minimum_charge = min(y_test.min(), y_pred.min())
maximum_charge = max(y_test.max(), y_pred.max())
plt.plot(
    [minimum_charge, maximum_charge],
    [minimum_charge, maximum_charge],
    color="red",
    linestyle="--",
    label="Perfect prediction",
)

plt.title("Actual vs. Predicted Insurance Charges")
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.legend()
plt.tight_layout()
plt.show()

## Conclusion

The notebook converts the categorical `smoker` variable with both `LabelEncoder()` and the current `OneHotEncoder()`/`ColumnTransformer` approach, uses the required 15% test size, trains a multiple linear regression model, and evaluates how well the available applicant characteristics predict insurance charges.